# Foosball SAC Agent — Kaggle Training v3

Train a **Soft Actor-Critic (SAC)** reinforcement learning agent to play foosball using:
- **MuJoCo** physics simulation () — foosmen limited to ±90° rotation
- **Stable-Baselines3** SAC implementation
- **Protagonist-Antagonist** self-play curriculum via 

**v3 changes vs v2:** foosmen rotation capped at ±π/2 (cannot go upside-down); kick window restricted to [17°–60°].

**Repo:** https://github.com/carlkaziboni/Foosball_CU.git
**Output:** models saved to  — downloadable as zip at the end.

---
Run cells **top to bottom**. GPU (P100/T4) auto-detected and hyperparameters scaled accordingly.


## 1. Install System & Python Dependencies

Installs headless OpenGL libraries, MuJoCo, Stable-Baselines3, and all supporting packages.
The repo is cloned (or pulled if already present) and added to .


In [ ]:
import subprocess, sys, os, shutil

def run(cmd):
    subprocess.check_call(cmd)

def pip(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

# ── System packages for headless OpenGL rendering ────────────────────────────
run(["apt-get", "install", "-y", "-q",
     "xvfb", "libgl1-mesa-glx", "libosmesa6", "libglfw3", "ffmpeg"])

# ── Python packages ────────────────────────────────────────────────────────────
pip(
    "mujoco",
    "stable-baselines3[extra]",
    "gymnasium",
    "shimmy>=0.2.0",
    "pyvirtualdisplay",
    "glfw",
    "tensorboard",
)

# ── Clone / update repo ────────────────────────────────────────────────────────
REPO_URL = "https://github.com/carlkaziboni/Foosball_CU.git"
REPO_DIR = "/kaggle/working/Foosball_CU"

# Always do a fresh shallow clone to guarantee we have the latest commit.
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
print(f"Repo cloned → {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install any project-level requirements
req_file = os.path.join(REPO_DIR, "requirements.txt")
if os.path.exists(req_file):
    pip("-r", req_file)
    print("requirements.txt installed ✓")

print("\nCell 1 done ✓")

## 2. Headless Rendering Setup & GLFW Patch

Sets , starts a virtual display, then **patches  to a no-op class** before  is ever imported.
This prevents GLFW window errors in Kaggle's headless kernel.


In [ ]:
import os, sys, warnings

# ── Headless env vars — must be set BEFORE any mujoco/glfw import ─────────────
os.environ["MUJOCO_GL"]         = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"

# ── Virtual display ────────────────────────────────────────────────────────────
from pyvirtualdisplay import Display
_display = Display(visible=False, size=(1280, 960))
_display.start()
os.environ["DISPLAY"] = ":0"
print("Virtual display started ✓")

# ── Silence gym deprecation warnings ──────────────────────────────────────────
warnings.filterwarnings("ignore", message=".*upgrade to Gymnasium.*")

# ── Clone / pull repo ─────────────────────────────────────────────────────────
import subprocess
REPO_URL = "https://github.com/carlkaziboni/Foosball_CU.git"
REPO_DIR = "/kaggle/working/Foosball_CU"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
    print(f"Repo cloned → {REPO_DIR} ✓")
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    print(f"Repo updated ✓")

# ── Create v3/asset symlink if it does not exist ──────────────────────────────
v3_asset = os.path.join(REPO_DIR, "foosball_sim", "v3", "asset")
v2_asset = os.path.join(REPO_DIR, "foosball_sim", "v2", "asset")
if not os.path.exists(v3_asset):
    os.symlink(v2_asset, v3_asset)
    print(f"Symlink created: v3/asset → v2/asset ✓")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Stub out the GLFW render mixin BEFORE importing FoosballEnv ───────────────
import types
_stub_mod_v3 = types.ModuleType("ai_agents.v3.gym.mujoco_table_render_mixin")
_stub_mod_v3.MujocoTableRenderMixin = type("MujocoTableRenderMixin", (), {})
sys.modules["ai_agents.v3.gym.mujoco_table_render_mixin"] = _stub_mod_v3
print("GLFW render mixin stubbed for v3 ✓")
print("Cell 2 done ✓")


## 3. Environment Factory

Imports  from **v3** (safe now that the mixin is patched) and defines the factory used everywhere.
 is **38-dim** continuous;  is **8-dim** continuous (4 rods × 2: linear + rotation).
Foosmen are capped at **±90° rotation** — no upside-down kicks.


In [ ]:
# Mixin is already patched — safe to import v3 FoosballEnv now
from stable_baselines3.common.monitor import Monitor
from ai_agents.v3.gym.full_information_protagonist_antagonist_gym import FoosballEnv

# ── Global antagonist slot — set to a loaded SAC model to enable self-play ─────
_current_antagonist = None  # Phase 1: None = solo training


def sac_foosball_env_factory(x=None):
    """Creates a monitored FoosballEnv (v3) using the current global antagonist."""
    env = FoosballEnv(antagonist=_current_antagonist)
    return Monitor(env, LOG_DIR)


MODEL_DIR = "/kaggle/working/models_v3"
LOG_DIR   = "/kaggle/working/logs_v3"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)

# Quick sanity check
_test_env = sac_foosball_env_factory()
obs, _ = _test_env.reset()
print(f"v3 env observation space : {_test_env.observation_space}")
print(f"v3 env action space      : {_test_env.action_space}")
print(f"Sample obs shape         : {obs.shape}")
_test_env.close()
print("Cell 3 done ✓")


## 4. GPU Detection & Stable SAC Agent

Detects available hardware and defines  — a subclass of  tuned for foosball v3.


In [ ]:
import torch
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from ai_agents.common.train.impl.sac_agent import SACFoosballAgent

# ── Detect device ──────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Training device : CUDA — {gpu_name}")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Training device : Apple Silicon MPS")
else:
    DEVICE = "cpu"
    print("Training device : CPU")


class StableSACAgent(SACFoosballAgent):
    """SAC agent tuned for foosball v3 (±90° rotation physics)."""

    def __init__(self, agent_id, environment_generator, model_dir, device=DEVICE):
        super().__init__(
            agent_id=agent_id,
            environment_generator=environment_generator,
            model_dir=model_dir,
            net_arch=[512, 512, 256],
            learning_rate=3e-4,
            buffer_size=300_000,
            batch_size=512,
            tau=0.005,
            gamma=0.99,
            use_sde=False,
            device=device,
        )


print("StableSACAgent (v3) defined ✓")
print("Cell 4 done ✓")


## 5. Training Configuration

Fresh start (no pretrained source — v3 physics differ from v2).

| Setting | GPU | CPU |
|---|---|---|
|  | 200 | 40 |
|  | 10 000 | 5 000 |
| **Total timesteps** | **2 000 000** | **200 000** |
|  | every epoch | every epoch |
| Timestep milestone saves | every 100 000 steps | every 25 000 steps |
| EvalCallback frequency | every 2 000 steps | every 2 000 steps |


In [ ]:
IS_GPU = DEVICE == "cuda"

# ── Auto-scale based on available hardware ─────────────────────────────────────
if IS_GPU:
    total_epochs           = 200
    epoch_timesteps        = 10_000
    cycle_timesteps        = 1_000
    milestone_step_freq    = 200_000   # save every 200k steps → 10 checkpoints over 2M ts
else:
    total_epochs           = 40
    epoch_timesteps        = 5_000
    cycle_timesteps        = 500
    milestone_step_freq    = 25_000    # save every 25k steps on CPU

# ── Override here if you want a custom run ─────────────────────────────────────
# total_epochs    = 200
# epoch_timesteps = 10_000

total_timesteps = total_epochs * epoch_timesteps

# ── Self-play curriculum ────────────────────────────────────────────────────────
self_play_reward_threshold = 500.0
self_play_reward_window    = 20
self_play_update_interval  = 5

# ── v3 starts fresh — no pretrained weights from v2 ───────────────────────────
PRETRAINED_SOURCE = None

print("=" * 54)
print(f"  Device              : {DEVICE.upper()}")
print(f"  Pretrained source   : {PRETRAINED_SOURCE or 'none (fresh start)'}")
print(f"  Epochs              : {total_epochs}")
print(f"  Timesteps / epoch   : {epoch_timesteps:,}")
print(f"  Total new timesteps : {total_timesteps:,}")
print(f"  Milestone save freq : every {milestone_step_freq:,} steps")
print(f"  Checkpoint / epoch  : every epoch")
print(f"  EvalCallback freq   : every 2 000 steps")
print(f"  Self-play threshold : mean reward ≥ {self_play_reward_threshold}")
print(f"  Model output dir    : {MODEL_DIR}")
print("=" * 54)
print("Cell 5 done ✓")


## 6. Agent Manager & Training Engine

 owns one .
 runs one epoch per call, saves checkpoints, and handles model reloading between epochs.


In [ ]:
from ai_agents.common.train.impl.generic_agent_manager import GenericAgentManager
from ai_agents.common.train.impl.single_player_training_engine import SinglePlayerTrainingEngine

# ── Agent manager ──────────────────────────────────────────────────────────────
agent_manager = GenericAgentManager(
    num_agents=1,
    environment_generator=sac_foosball_env_factory,
    agent_class=StableSACAgent,
)
agent_manager.initialize_training_agents()
agent_manager.initialize_frozen_best_models()

print("Agent manager initialised ✓")
print("Cell 6 done ✓")


## 7. Run Training — Dense Checkpoints + Reward-Triggered Self-Play

**Checkpoints saved:**
-  — whenever EvalCallback finds a new best (every 2 000 steps)
-  — **every epoch** (full coverage of training history)
-  — every  steps (100k on GPU, 25k on CPU)
-  — per-epoch reward log for offline plotting

**Fresh v3 run** — training from timestep 0.
Target: **2 000 000 timesteps** on GPU.


In [ ]:
import time, csv, math
from stable_baselines3 import SAC

FROZEN_CHECKPOINT = os.path.join(MODEL_DIR, "0", "sac", "best_model", "model.zip")
MONITOR_CSV       = os.path.join(LOG_DIR, "monitor.monitor.csv")
REWARD_LOG_CSV    = os.path.join(LOG_DIR, "training_rewards.csv")

# ── Helpers ────────────────────────────────────────────────────────────────────
def read_recent_mean_reward(csv_path, window=20):
    if not os.path.exists(csv_path):
        return None
    rewards = []
    try:
        with open(csv_path, "r") as f:
            reader = csv.DictReader(row for row in f if not row.startswith("#"))
            for row in reader:
                rewards.append(float(row["r"]))
    except Exception:
        return None
    if not rewards:
        return None
    return sum(rewards[-window:]) / len(rewards[-window:])


def save_checkpoint(agent, path_suffix, total_ts):
    """Save model to MODEL_DIR/0/sac/<path_suffix>/model.zip"""
    ck_path = os.path.join(MODEL_DIR, "0", "sac", path_suffix)
    os.makedirs(ck_path, exist_ok=True)
    agent.model.save(os.path.join(ck_path, "model"))
    with open(os.path.join(ck_path, "info.txt"), "w") as f:
        f.write(f"total_timesteps_at_save: {total_ts}
")
        f.write(f"path_suffix: {path_suffix}
")


def log_epoch_reward(epoch, total_ts, mean_r, phase):
    """Append one row to the per-epoch reward CSV."""
    write_header = not os.path.exists(REWARD_LOG_CSV)
    with open(REWARD_LOG_CSV, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["epoch", "total_timesteps", "mean_reward", "phase"])
        w.writerow([epoch, total_ts, f"{mean_r:.3f}" if mean_r is not None else "n/a", phase])


# ── Training state ─────────────────────────────────────────────────────────────
wall_start = time.time()
global _current_antagonist
_current_antagonist = None

in_self_play     = False
self_play_epoch  = 0
epochs_completed = 0
total_ts_so_far  = 0
next_milestone_ts = milestone_step_freq
pretrained_ts    = 0   # v3: fresh start

print(f"Starting v3 training — {total_epochs} epochs  ({total_timesteps:,} timesteps)
")

for epoch in range(1, total_epochs + 1):

    # ── Refresh antagonist if in self-play ────────────────────────────────────
    if in_self_play and (self_play_epoch % self_play_update_interval == 0):
        try:
            _current_antagonist = SAC.load(FROZEN_CHECKPOINT, device=DEVICE)
            print(f"  [Epoch {epoch}] Antagonist refreshed ✓")
        except Exception as e:
            print(f"  [Epoch {epoch}] Antagonist reload failed ({e})")

    # ── Train one epoch ────────────────────────────────────────────────────────
    ep_engine = SinglePlayerTrainingEngine(
        agent_manager=agent_manager,
        environment_generator=sac_foosball_env_factory,
    )
    ep_engine.train(
        total_epochs=1,
        epoch_timesteps=epoch_timesteps,
        cycle_timesteps=cycle_timesteps,
    )
    epochs_completed  += 1
    total_ts_so_far   += epoch_timesteps
    if in_self_play:
        self_play_epoch += 1

    protagonist_agent = agent_manager.get_training_agents()[0]

    # ── Dense epoch checkpoint ────────────────────────────────────────────────
    save_checkpoint(protagonist_agent, f"epoch_{epoch:04d}", total_ts_so_far)
    print(f"  💾 epoch_{epoch:04d} checkpoint saved")

    # ── Milestone timestep checkpoint ─────────────────────────────────────────
    while total_ts_so_far >= next_milestone_ts:
        label = f"timestep_{next_milestone_ts // 1000}K"
        save_checkpoint(protagonist_agent, label, total_ts_so_far)
        print(f"  🎯 {label} checkpoint saved  (total {total_ts_so_far:,} ts)")
        next_milestone_ts += milestone_step_freq

    # ── Read mean reward & log ─────────────────────────────────────────────────
    mean_r      = read_recent_mean_reward(MONITOR_CSV, window=self_play_reward_window)
    phase_label = "SELF-PLAY" if in_self_play else "SOLO"
    r_str       = f"{mean_r:.1f}" if mean_r is not None else "n/a"
    log_epoch_reward(epoch, total_ts_so_far, mean_r, phase_label)

    elapsed = time.time() - wall_start
    h, rem  = divmod(int(elapsed), 3600)
    m, s    = divmod(rem, 60)
    print(f"  [Epoch {epoch:>3}/{total_epochs}] {phase_label:<9}  "
          f"mean_r={r_str:<8}  total_ts={total_ts_so_far:,}  "
          f"elapsed={h:02d}h{m:02d}m{s:02d}s")

    # ── Self-play phase transition ─────────────────────────────────────────────
    if not in_self_play and mean_r is not None and mean_r >= self_play_reward_threshold:
        in_self_play    = True
        self_play_epoch = 0
        print(f"
{'='*62}")
        print(f"  *** Reward threshold reached ({mean_r:.1f} ≥ {self_play_reward_threshold}) ***")
        print(f"  Switching to SELF-PLAY at epoch {epoch + 1}")
        print(f"{'='*62}
")
        try:
            _current_antagonist = SAC.load(FROZEN_CHECKPOINT, device=DEVICE)
            print(f"  Initial antagonist loaded ✓")
        except Exception as e:
            print(f"  Could not load initial antagonist ({e})")

# ── Final cleanup ──────────────────────────────────────────────────────────────
_current_antagonist = None

elapsed = time.time() - wall_start
h, rem  = divmod(int(elapsed), 3600)
m, s    = divmod(rem, 60)
print("
" + "=" * 62)
print(f"Training complete ✓  ({h:02d}h {m:02d}m {s:02d}s)")
print(f"Total timesteps  : {total_ts_so_far:,}")
phase_str = f"triggered at reward ≥ {self_play_reward_threshold}" if in_self_play else f"never triggered (max r={r_str})"
print(f"Self-play        : {phase_str}")
print(f"Reward log       : {REWARD_LOG_CSV}")
print(f"Models saved to  : {MODEL_DIR}")
print("=" * 62)


## 8. Inspect Saved Checkpoints

Lists every file written under  so you can verify checkpoints are present before downloading.


In [ ]:
import glob

print(f"Checkpoint inventory under {MODEL_DIR}:\n")

# Group by checkpoint folder
ck_dirs = sorted(set(
    os.path.dirname(f)
    for f in glob.glob(f"{MODEL_DIR}/**/*", recursive=True)
    if os.path.isfile(f)
))

for d in ck_dirs:
    files = [f for f in glob.glob(os.path.join(d, "*")) if os.path.isfile(f)]
    total_kb = sum(os.path.getsize(f) for f in files) / 1024
    label    = d.replace(MODEL_DIR + "/", "")
    # Read info.txt if present
    info_path = os.path.join(d, "info.txt")
    info_str  = ""
    if os.path.exists(info_path):
        with open(info_path) as f:
            info_str = "  " + f.read().strip().replace("\n", ", ")
    print(f"  {label:<40}  {total_kb:>7.0f} KB{info_str}")

# Print reward log summary if it exists
print()
if os.path.exists(REWARD_LOG_CSV):
    import csv as _csv
    rows = []
    with open(REWARD_LOG_CSV) as f:
        rows = list(_csv.DictReader(f))
    if rows:
        rewards = [float(r["mean_reward"]) for r in rows if r["mean_reward"] != "n/a"]
        print(f"Reward log  : {len(rows)} epochs recorded")
        if rewards:
            print(f"  best mean : {max(rewards):.1f}")
            print(f"  last mean : {rewards[-1]:.1f}")
else:
    print("(no reward log yet)")

print("\nCell 8 done ✓")

## 9. Evaluate Trained Agent

Loads the best saved model and runs  deterministic episodes.
Prints per-episode reward and a summary (mean / min / max).


In [ ]:
NUM_EVAL_EPISODES = 10

# Re-initialise frozen models so they pick up the latest saved checkpoint
agent_manager.initialize_frozen_best_models()
protagonist = agent_manager.get_frozen_best_models()[0]

eval_env = sac_foosball_env_factory()
episode_rewards = []

print(f"Evaluating trained agent — {NUM_EVAL_EPISODES} episodes (deterministic)\n")

for ep in range(NUM_EVAL_EPISODES):
    obs, _ = eval_env.reset()
    done = False
    total_reward = 0.0
    steps = 0

    while not done:
        # protagonist is a SACFoosballAgent — its predict() already unwraps (action, state)
        # and returns just the action array directly
        action = protagonist.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated

    episode_rewards.append(total_reward)
    print(f"  Episode {ep + 1:>2}/{NUM_EVAL_EPISODES}  reward: {total_reward:>8.2f}  steps: {steps}")

eval_env.close()

mean_r = sum(episode_rewards) / len(episode_rewards)
print(f"\n{'─' * 40}")
print(f"  Mean reward : {mean_r:.2f}")
print(f"  Min  reward : {min(episode_rewards):.2f}")
print(f"  Max  reward : {max(episode_rewards):.2f}")
print(f"{'─' * 40}")
print("Cell 9 done ✓")


## 9.5. Plot Training Reward Curve

Reads  (written every epoch during Cell 7) and plots the mean reward progression.


In [ ]:
import csv, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

if not os.path.exists(REWARD_LOG_CSV):
    print("No reward log found — run Cell 7 first.")
else:
    rows = []
    with open(REWARD_LOG_CSV) as f:
        rows = [r for r in csv.DictReader(f) if r["mean_reward"] != "n/a"]

    if not rows:
        print("Reward log is empty.")
    else:
        epochs   = [int(r["epoch"])          for r in rows]
        ts       = [int(r["total_timesteps"]) for r in rows]
        rewards  = [float(r["mean_reward"])   for r in rows]
        phases   = [r["phase"]                for r in rows]

        # Rolling mean (window=10)
        def roll(arr, w=10):
            return [np.mean(arr[max(0,i-w+1):i+1]) for i in range(len(arr))]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Training Reward Progression", fontsize=13, fontweight="bold")

        for ax, x_vals, x_label in [
            (axes[0], epochs, "Epoch"),
            (axes[1], ts,     "Total Timesteps"),
        ]:
            # Shade solo vs self-play regions
            in_sp = False
            for i, (xv, ph) in enumerate(zip(x_vals, phases)):
                if ph == "SELF-PLAY" and not in_sp:
                    ax.axvline(xv, color="purple", ls="--", lw=1.2, alpha=0.7,
                               label="Self-play starts")
                    in_sp = True

            ax.plot(x_vals, rewards, "o", ms=3, alpha=0.35, color="#4C9BE8")
            ax.plot(x_vals, roll(rewards), "-", lw=2.2, color="#1a6bbf",
                    label="Rolling mean (w=10)")
            ax.set_xlabel(x_label)
            ax.set_ylabel("Mean Episode Reward")
            ax.set_title(f"Reward vs {x_label}")
            ax.grid(alpha=0.3)
            ax.legend(fontsize=9)

        plt.tight_layout()
        plot_path = os.path.join(LOG_DIR, "training_reward_curve.png")
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        print(f"Plot saved → {plot_path}")

        print(f"\nEpochs logged  : {len(rows)}")
        print(f"Best mean r    : {max(rewards):.1f}  (epoch {epochs[np.argmax(rewards)]})")
        print(f"Last mean r    : {rewards[-1]:.1f}")
        sp_epochs = sum(1 for p in phases if p == "SELF-PLAY")
        print(f"Self-play eps  : {sp_epochs}/{len(rows)}")
print("Cell 9.5 done ✓")

## 10. Package & Download Models

Zips  into .
The archive is visible in the Kaggle **Output** tab — click the file to download.


In [ ]:
import shutil, glob

ARCHIVE_PATH = "/kaggle/working/foosball_sac_models_v3"

shutil.make_archive(
    base_name=ARCHIVE_PATH,
    format="zip",
    root_dir="/kaggle/working",
    base_dir="models_v3",
)

zip_path = f"{ARCHIVE_PATH}.zip"
zip_size_mb = os.path.getsize(zip_path) / 1e6

print(f"Archive created  : {zip_path}")
print(f"Archive size     : {zip_size_mb:.2f} MB")
print(f"
Contents:")
for f in sorted(glob.glob(f"{MODEL_DIR}/**/*", recursive=True)):
    if os.path.isfile(f):
        kb = os.path.getsize(f) / 1024
        print(f"  {f}  ({kb:.1f} KB)")

print("
Download from the Kaggle Output tab ✓")
